In [10]:
import pandas as pd

# Step 1: Read the CSV file
# df = pd.read_csv('log_B.csv')
# df = pd.read_csv('log_B_new.csv')
df = pd.read_excel('log_B.xlsx')

df['del_G_eV'] = df['del G (eV) Paul'].fillna(df['del G (eV) Azida']).fillna(df['del G (eV) Other']).fillna(df['del G (eV) Other'])
df['log_B'] = df["log_B Paul's Handbook"].fillna(df['log_B Azida Avg'])

df = df[~df['ligand'].str.contains('NH4', na=False)]
df = df[~df['ligand'].str.contains('OH', na=False)]
df = df[~df['ligand'].str.contains('Cl', na=False)]
# df = df[~df['ligand'].str.contains('CN', na=False)]


new_df = df[['ligand', 'metal_ion', 'n_metal', 'n_complex','G_ligand (kJ/mol)','G_metal (kJ/mol)', 'signed_metal_ion', 'del_G_eV','log_B']]
new_df
new_df=new_df.dropna(how='any')
new_df


,ligand,metal_ion,n_metal,n_complex,G_ligand (kJ/mol),G_metal (kJ/mol),signed_metal_ion,del_G_eV,log_B
0,NH3,Ag+,1,1,-26.63,77.107,Ag[1+],0.321990,3.40
1,NH3,Ag+,1,2,-26.63,77.107,Ag[1+],-0.190677,7.40
2,NH3,Au+,1,2,-26.63,176.000,Au[1+],-0.325391,27.00
3,NH3,Au3+,1,4,-26.63,440.000,Au[3+],1.681275,30.00
4,NH3,Ca2+,1,1,-26.63,-553.540,Ca[2+],-6.001205,-0.20
...,...,...,...,...,...,...,...,...,...
141,CN[1-],Zn2+,1,4,172.40,-147.060,Zn[2+],4.633763,16.72
142,CN[1-],Pt2+,1,4,172.40,254.800,Pt[2+],5.646346,70.00
144,NO2[1-],Cu2+,1,1,-32.20,65.490,Cu[2+],0.274026,1.20
145,NO2[1-],Cu2+,1,2,-32.20,65.490,Cu[2+],-0.072720,1.42


In [11]:
new_df['ligand']
new_df[new_df["metal_ion"] == "Pd2+"]#["ligand"].unique()


,ligand,metal_ion,n_metal,n_complex,G_ligand (kJ/mol),G_metal (kJ/mol),signed_metal_ion,del_G_eV,log_B
56,NH3,Pd2+,1,4,-26.630,176.5,Pd[2+],-1.215377,30.500000
99,Gly[1-],Pd2+,1,1,-314.833,172.4,Pd[2+],-2.015814,9.120000
100,Gly[1-],Pd2+,1,2,-314.833,172.4,Pd[2+],-5.777604,17.550000
138,CN[1-],Pd2+,1,1,172.400,176.5,Pd[2+],2.995274,10.492719
139,CN[1-],Pd2+,1,4,172.400,176.5,Pd[2+],5.290407,62.300000
140,CN[1-],Pd2+,1,5,172.400,176.5,Pd[2+],8.083042,45.300000


In [12]:
import pandas as pd
import re

def generate_species(row):
    signed_metal = row['signed_metal_ion'].split('[')
    ligand_part = row['ligand'].split('[')[0]
    
    # Handle n_metal display
    n_metal = int(row['n_metal'])  # Ensure n_metal is an integer
    n_metal_part = str(n_metal) if n_metal != 1 else ''
    
    # Extract the number and charge from signed_metal_ion_part
    if len(signed_metal) > 1:
        ion_part = signed_metal[1].strip(']')  # Remove closing bracket
        # Use regular expression to extract the number and charge
        match = re.match(r'(\d+)([+-])', ion_part)
        if match:
            number = int(match.group(1)) * n_metal  # Multiply the number by n_metal
            charge_sign = match.group(2)  # Extract the sign
            charge_multiplier = 1 if charge_sign == '+' else -1  # Apply the charge as multiplier
            
            result = number * charge_multiplier  # Multiply the number by the charge
 
            signed_metal_ion_part = f'[{result}{charge_sign}]'
 
        else:
            signed_metal_ion_part = ''  # Default case if not matched
    else:
        signed_metal_ion_part = ''
    
    # Concatenate the species name
    species = signed_metal[0] + n_metal_part + '(' + ligand_part + ')' + str(int(row['n_complex'])) + signed_metal_ion_part
    
    return species

# Apply the function to each row
new_df['species'] = new_df.apply(generate_species, axis=1)
new_df['metal'] = new_df['signed_metal_ion'].str.split('[').str[0]

# Display the updated DataFrame
new_df 


,ligand,metal_ion,n_metal,n_complex,G_ligand (kJ/mol),G_metal (kJ/mol),signed_metal_ion,del_G_eV,log_B,species,metal
0,NH3,Ag+,1,1,-26.63,77.107,Ag[1+],0.321990,3.40,Ag(NH3)1[1+],Ag
1,NH3,Ag+,1,2,-26.63,77.107,Ag[1+],-0.190677,7.40,Ag(NH3)2[1+],Ag
2,NH3,Au+,1,2,-26.63,176.000,Au[1+],-0.325391,27.00,Au(NH3)2[1+],Au
3,NH3,Au3+,1,4,-26.63,440.000,Au[3+],1.681275,30.00,Au(NH3)4[3+],Au
4,NH3,Ca2+,1,1,-26.63,-553.540,Ca[2+],-6.001205,-0.20,Ca(NH3)1[2+],Ca
...,...,...,...,...,...,...,...,...,...,...,...
141,CN[1-],Zn2+,1,4,172.40,-147.060,Zn[2+],4.633763,16.72,Zn(CN)4[2+],Zn
142,CN[1-],Pt2+,1,4,172.40,254.800,Pt[2+],5.646346,70.00,Pt(CN)4[2+],Pt
144,NO2[1-],Cu2+,1,1,-32.20,65.490,Cu[2+],0.274026,1.20,Cu(NO2)1[2+],Cu
145,NO2[1-],Cu2+,1,2,-32.20,65.490,Cu[2+],-0.072720,1.42,Cu(NO2)2[2+],Cu


In [13]:
import os

# Ensure the 'data' directory exists
if not os.path.exists('data'):
    os.makedirs('data')

# Save the DataFrame as a JSON file
new_df.to_json('data/metal_complex_del_G.json', orient='records', indent=4)

# Confirm save
print("DataFrame saved to 'data/metal_complex_del_G.json'")


DataFrame saved to 'data/metal_complex_del_G.json'


In [1]:
import pandas as pd

meng_Ni_NH3_eng = {}

meng_Ni_NH3_eng['Ni(NH3)[2+]']= -87814 #j/mol
meng_Ni_NH3_eng['Ni(NH3)2[2+]'] =-126842
meng_Ni_NH3_eng['Ni(NH3)3[2+]'] =-162981
meng_Ni_NH3_eng['Ni(NH3)4[2+]']= -196022
meng_Ni_NH3_eng['Ni(NH3)5[2+]']=-226470
meng_Ni_NH3_eng['Ni(NH3)6[2+]']= -252931
meng_Ni_NH3_eng['NH3']= -26633
meng_Ni_NH3_eng['Au[3+]']= 433462.4
meng_Ni_NH3_eng['Au(NH3)4[3+]']= 64437.784


df = pd.DataFrame(list(meng_Ni_NH3_eng.items()), columns=['formula', 'del_G(J/mol)'])
df

,formula,del_G(J/mol)
0,Ni(NH3)[2+],-87814.000
1,Ni(NH3)2[2+],-126842.000
2,Ni(NH3)3[2+],-162981.000
3,Ni(NH3)4[2+],-196022.000
4,Ni(NH3)5[2+],-226470.000
5,Ni(NH3)6[2+],-252931.000
6,NH3,-26633.000
7,Au[3+],433462.400
8,Au(NH3)4[3+],64437.784


In [2]:
import numpy as np

# Constants
R = 8.314  # J/(mol·K), universal gas constant
T = 298.15  # K, temperature (standard room temperature)
J_TO_EV = 1.60218e-19  # Conversion factor from Joules to electron volts (J/eV)
AVOGADRO_NUMBER = 6.022e23  # Avogadro's number (molecules/mol)

def convert_joules_to_ev(joules_per_mol):
    """
    Convert energy from Joules per molecule to eV.
    
    Args:
        joules_per_molecule (float): Energy in Joules per molecule.
    
    Returns:
        float: Energy in electron volts (eV).
    """
    joules_per_molecule = joules_per_mol / AVOGADRO_NUMBER
    ev_per_molecule = joules_per_molecule / J_TO_EV
    return ev_per_molecule


df['del_G(eV)'] = convert_joules_to_ev(df['del_G(J/mol)'])
df['del_G(kJ/mol)'] = df['del_G(J/mol)']/1000
df

,formula,del_G(J/mol),del_G(eV),del_G(kJ/mol)
0,Ni(NH3)[2+],-87814.000,-0.910147,-87.814000
1,Ni(NH3)2[2+],-126842.000,-1.314653,-126.842000
2,Ni(NH3)3[2+],-162981.000,-1.689215,-162.981000
3,Ni(NH3)4[2+],-196022.000,-2.031668,-196.022000
4,Ni(NH3)5[2+],-226470.000,-2.347246,-226.470000
5,Ni(NH3)6[2+],-252931.000,-2.621501,-252.931000
6,NH3,-26633.000,-0.276037,-26.633000
7,Au[3+],433462.400,4.492617,433.462400
8,Au(NH3)4[3+],64437.784,0.667865,64.437784


In [3]:
2.717/(4*0.0591)

11.493231810490695

In [4]:
convert_joules_to_ev(-T*R*np.log(1e46))

-2.721236558825371

In [5]:
kB = 8.6173e-5 #eV/K 
T*kB*np.log(10)

0.0591591213349184

In [6]:
import json 

filtered_df = df[df['formula'].str.contains('Ni')]

source_citation = "Protonation and Complex Formation Equilibrium constants"
json_data_list = []

# Loop through each row and create the dictionary
for idx, row in filtered_df.iterrows():
    # Build the dictionary for each row
    row_data = {
        "Reference solid energy": -4.696,
        "Major_Elements": [r"Ni"],
#         "Major_Elements": [row['signed_metal_ion'].split('[')[0]],
        "Energy": row['del_G(eV)'],
        "Source": source_citation,
        "Reference Solid":  "Ni(HO)2",
        "Name": row['formula']
    }
    
    # Append the dictionary to the list
    json_data_list.append(row_data)

# Define the output JSON filename
output_filename = "Meng_Ni_NH3_aqueous_ion_entries.json"
# Write the list of dictionaries to a single JSON file
with open(output_filename, 'w') as f:
    json.dump(json_data_list, f, indent=4)

print(f"Created {output_filename}")

Created Meng_Ni_NH3_aqueous_ion_entries.json


In [1]:
def kcal_per_mol_to_ev_per_molecule(kcal_per_mol):
    # Conversion factor from kcal/mol to eV/molecule
    conversion_factor = 0.0433641153087705
    return kcal_per_mol * conversion_factor

# Example usage:
kcal_value = 10  # Replace with your kcal/mol value
ev_value = kcal_per_mol_to_ev_per_molecule(kcal_value)
print(f"{kcal_value} kcal/mol is {ev_value} eV/molecule")


10 kcal/mol is 0.43364115308770496 eV/molecule


In [5]:
Zr_NH3 = [72.1,74.9,57.6, 56.5, 47.1, 53.2,44.5]

for i in Zr_NH3:
    print(kcal_per_mol_to_ev_per_molecule(i))


3.1265527137623526
3.2479722366269104
2.4977730417851807
2.450072514945533
2.0424498310430903
2.3069709344265905
1.929703131240287


In [5]:
TiGly2 = -163.34
kcal_per_mol_to_ev_per_molecule(TiGly2)

-7.083094594534573

In [10]:
import pandas as pd

# Step 1: Read the CSV file
df = pd.read_csv('log_B_Ni_only.csv')
# df = pd.read_csv('log_B.csv')

df=df.dropna()
df['B'] = 10**df['log_B'] # 
df = df.drop('metal_ion', axis = 1)
# df = df.drop('log_B', axis = 1)

In [12]:
import numpy as np

# Constants
R = 8.314  # J/(mol·K), universal gas constant
T = 298.15  # K, temperature (standard room temperature)
J_TO_EV = 1.60218e-19  # Conversion factor from Joules to electron volts (J/eV)
AVOGADRO_NUMBER = 6.022e23  # Avogadro's number (molecules/mol)

def calculate_delta_g_per_molecule(K, T=298.15):
    """
    Calculate Gibbs free energy change (ΔG) per molecule from the formation constant (K).
    
    Args:
        K (float): The formation constant (dimensionless).
        T (float): Temperature in Kelvin. Default is 298.15 K.
    
    Returns:
        float: Gibbs free energy change per molecule in J.
    """
    delta_g_j_per_mol = -R * T * np.log(K)
    
    delta_g_j_per_molecule = delta_g_j_per_mol / AVOGADRO_NUMBER
    return delta_g_j_per_molecule

def convert_joules_to_ev(joules_per_molecule):
    """
    Convert energy from Joules per molecule to eV.
    
    Args:
        joules_per_molecule (float): Energy in Joules per molecule.
    
    Returns:
        float: Energy in electron volts (eV).
    """
    ev_per_molecule = joules_per_molecule / J_TO_EV
    return ev_per_molecule


df['rxn_G(eV)'] = convert_joules_to_ev(calculate_delta_g_per_molecule(df['B'], T))
df['rxn_G(kJ/mol)'] = df['rxn_G(eV)']*J_TO_EV*AVOGADRO_NUMBER/1000

E_Ni_ion = -45.6 # kJ/mol
df['del_G(kJ/mol)'] = df['rxn_G(kJ/mol)'] - 
# df[:10]
# df['log10_'] = df['del_G(eV)']*J_TO_EV*AVOGADRO_NUMBER/1000


,ligand,complex,log_B,B,rxn_G(eV),rxn_G(kJ/mol)
0,NH3,l,2.75,5.623413e+02,-0.162683,-15.696153
1,NH3,l2,4.95,8.912509e+04,-0.292829,-28.253075
2,NH3,l3,6.64,4.365158e+06,-0.392805,-37.899074
3,NH3,l4,7.79,6.165950e+07,-0.460835,-44.462920
4,NH3,l5,8.50,3.162278e+08,-0.502837,-48.515381
5,NH3,l6,8.49,3.090295e+08,-0.502246,-48.458304
6,N2H4,l,3.18,1.513561e+03,-0.188120,-18.150460
7,OH-,L,4.60,3.981072e+04,-0.272124,-26.255383
8,glycine,l,5.80,6.309573e+05,-0.343112,-33.104613
9,glycine,l2,10.70,5.011872e+10,-0.632983,-61.072303


In [16]:
convert_joules_to_ev(-45.6*1000/AVOGADRO_NUMBER)

-0.4726207503419069

In [39]:
-79.497 *1000/J_TO_EV/ AVOGADRO_NUMBER# kj/mol

-0.8239458725861968

In [40]:
import re

# Function to extract the charge from metal_ion and ligand
def extract_charge(ion):
    match = re.search(r'\[(\d*)([+-])\]', ion)
    if match:
        charge = match.group(1)
        sign = match.group(2)
        if charge == '':
            charge =1
        if sign == '+':
            return int(charge)
        else:
            return -int(charge)
    return 0

# Function to calculate the total charge and format the new string
def create_combined(row):
    metal_charge = extract_charge(row['signed_metal_ion'])
    ligand_charge = extract_charge(row['ligand'])
    
    # Extract number of ligands from the complex
    ligand_count_match = re.search(r'([Ll])(\d*)', row['complex'])
    number = ligand_count_match.group(2)
    ligand_count_string = ''
    if number == '':
        ligand_count=1
    else:
        ligand_count =int(number)
        ligand_count_string = number
    
    # Total charge = metal charge + ligand charge * number of ligands
    total_charge = metal_charge + (ligand_charge * ligand_count)
    
    # Determine the sign of the total charge
    charge_sign = '+' if total_charge > 0 else '-'
    if total_charge == 1 or total_charge == -1:
        charge_str = f"[{charge_sign}]"
    else:
        
        charge_str = f"[{abs(total_charge)}{charge_sign}]"
    if total_charge == 0:
        charge_str=''
    
    # Create the final string
    return f"{row['signed_metal_ion'].split('[')[0]}({row['ligand'].split('[')[0]}){ligand_count_string}{charge_str}"

# Apply the function to create the new column
df['combined'] = df.apply(create_combined, axis=1)
df=df.drop('complex', axis = 1)
df=df.drop('ligand', axis = 1)



KeyError: 'signed_metal_ion'

In [41]:
df

,ligand,complex,log_B,B,del_G(eV),del_G(kJ/mol)
0,NH3,l,2.75,5.623413e+02,-0.162683,-15.696153
1,NH3,l2,4.95,8.912509e+04,-0.292829,-28.253075
2,NH3,l3,6.64,4.365158e+06,-0.392805,-37.899074
3,NH3,l4,7.79,6.165950e+07,-0.460835,-44.462920
4,NH3,l5,8.50,3.162278e+08,-0.502837,-48.515381
5,NH3,l6,8.49,3.090295e+08,-0.502246,-48.458304
6,N2H4,l,3.18,1.513561e+03,-0.188120,-18.150460
7,OH-,L,4.60,3.981072e+04,-0.272124,-26.255383
8,glycine,l,5.80,6.309573e+05,-0.343112,-33.104613
9,glycine,l2,10.70,5.011872e+10,-0.632983,-61.072303


In [124]:
import json 

filtered_df = df[df['signed_metal_ion'].str.contains('Ni')]

source_citation = "Protonation and Complex Formation Equilibrium constants"
json_data_list = []

# Loop through each row and create the dictionary
for idx, row in filtered_df.iterrows():
    # Build the dictionary for each row
    row_data = {
        "Reference solid energy": -4.696,
        "Major_Elements": [row['signed_metal_ion'].split('[')[0]],
        "Energy": row['del_G(eV)'],
        "Source": source_citation,
        "Reference Solid":  "Ni(HO)2",
        "Name": row['combined']
    }
    
    # Append the dictionary to the list
    json_data_list.append(row_data)

# Define the output JSON filename
output_filename = "Ni_NH3_aqueous_ion_entries.json"
# Write the list of dictionaries to a single JSON file
with open(output_filename, 'w') as f:
    json.dump(json_data_list, f, indent=4)

print(f"Created {output_filename}")

Created Ni_NH3_aqueous_ion_entries.json


In [32]:
G = 207000 #j/mol
np.log10(np.exp(-G/R/T))



-36.266848901548784

In [36]:
G = -46400 #j/mol
G/J_TO_EV/AVOGADRO_NUMBER

-0.4809123424531684